In this notebook, we will merge the yamanishi datasets: enzymes, ion channels, GPCRs, and nuclear receptors into a single CSV file.

We will not merge the drugbank dataset with Yamanishi, as they imply on doing the same thing in their paper.



In [54]:
import pandas as pd
from pathlib import Path

DATA_DIR_YAMANISHI = Path("../data/positive dti datasets/yamanishi/final yamanishi") 
DATA_DIR_DRUGBANK = Path("../data/positive dti datasets/drugbank/drugbank beginning dataset") 

files_yamanishi = {
     "enzymes.csv",
    "gpcr.csv",
    "ion_channel.csv",
    "nuclear_receptor.csv",
}

#get the dataframes from yamanishi
dataframes_yamanishi = []
for file in files_yamanishi:
    csv_path = DATA_DIR_YAMANISHI / file
    df = pd.read_csv(csv_path)
    dataframes_yamanishi.append(df) 
#merge yamanishi dataframes
merged_yamanishi_df = pd.concat(dataframes_yamanishi, ignore_index=True)


merged_yamanishi_df.shape



(5127, 3)

In [55]:
#getting rid of drug target duplicates in yamanishi
merged_yamanishi_df = merged_yamanishi_df.drop_duplicates(subset=['drug_id','target_id'])
merged_yamanishi_df.shape
merged_yamanishi_df.columns
merged_yamanishi_df.head()

#save the merged yamanishi dataframe as a csv
merged_yamanishi_csv_path = DATA_DIR_YAMANISHI / "yamanishi_merged.csv"
merged_yamanishi_df.to_csv(merged_yamanishi_csv_path, index=False)

In [56]:

#get df drugbank
csv_path = DATA_DIR_DRUGBANK / "drugbank_dti_with_kegg.csv"
df_drugbank = pd.read_csv(csv_path)

df_drugbank.shape
df_drugbank.columns

Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id',
       'protein_uniprot_id', 'kegg_protein_id', 'protein_sequence', 'organism',
       'label'],
      dtype='object')

In [57]:
#we wanna delete all rows in drugbank that have invalid SMILES and invalid sequences
import pandas as pd
from rdkit import Chem
from Bio.SeqUtils import IUPACData
from tqdm import tqdm


def is_valid_smiles(smiles):
    if not isinstance(smiles, str) or smiles.strip() == "":
        return False
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def is_valid_protein(seq):
    if not isinstance(seq, str) or seq.strip() == "":
        return False
    seq = seq.upper().strip()
    
    # Remove FASTA header if present
    if seq.startswith(">"):
        seq = "\n".join(seq.split("\n")[1:])
    
    seq = seq.replace("\n", "").replace(" ", "")
    
    # Reject if too short to be real protein
    if len(seq) < 20:
        return False

    return all(aa in VALID_AA for aa in seq)

tqdm.pandas()

df_drugbank["valid_smiles"] = df_drugbank["smiles"].progress_apply(is_valid_smiles)
df_drugbank["valid_protein"] = df_drugbank["protein_sequence"].progress_apply(is_valid_protein)

print("Before cleaning:", df_drugbank.shape)




 11%|█         | 2679/24525 [00:00<00:02, 8873.27it/s][10:34:28] Explicit valence for atom # 0 N, 4, is greater than permitted
[10:34:28] Explicit valence for atom # 0 N, 4, is greater than permitted
[10:34:28] Explicit valence for atom # 0 N, 4, is greater than permitted
[10:34:28] Explicit valence for atom # 0 N, 4, is greater than permitted
 30%|███       | 7477/24525 [00:00<00:01, 9300.26it/s][10:34:29] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[10:34:29] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[10:34:29] Explicit valence for atom # 13 Cl, 5, is greater than permitted
 51%|█████     | 12440/24525 [00:01<00:01, 9010.49it/s][10:34:29] Explicit valence for atom # 0 O, 3, is greater than permitted
[10:34:29] Explicit valence for atom # 3 N, 4, is greater than permitted
[10:34:29] Unusual charge on atom 0 number of radical electrons set to zero
[10:34:29] Explicit valence for atom # 4 F, 2, is greater than permitted
[10:34:29] Explicit valen

Before cleaning: (24525, 11)


In [58]:
df_drugbank = df_drugbank[
    (df_drugbank["valid_smiles"] == True) &
    (df_drugbank["valid_protein"] == True)
].copy()

print("After cleaning:", df_drugbank.shape)
df_drugbank

After cleaning: (24451, 11)


,drugbank_id,drug_name,smiles,kegg_drug_id,protein_uniprot_id,kegg_protein_id,protein_sequence,organism,label,valid_smiles,valid_protein
0,DB00006,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,D03136,P00734,NaN,>lcl|BSEQ0016004|Prothrombin\nMAHVRGLQLPGCLALA...,Humans,1,True,True
1,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,D00573,P22888,NaN,>lcl|BSEQ0036957|Lutropin-choriogonadotropic h...,Humans,1,True,True
2,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,D00573,P01148,NaN,>lcl|BSEQ0053339|Progonadoliberin-1\nMKPIQKLLA...,Humans,1,True,True
3,DB00014,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,D00573,P30968,NaN,>lcl|BSEQ0000405|Gonadotropin-releasing hormon...,Humans,1,True,True
4,DB00027,Gramicidin D,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,D04369,P0AC13,NaN,>lcl|BSEQ0001602|Dihydropteroate synthase\nMKL...,Escherichia coli (strain K12),1,True,True
...,...,...,...,...,...,...,...,...,...,...,...
24520,DB19353,Benzgalantamine,[H][C@]12C[C@@H](OC(=O)C3=CC=CC=C3)C=C[C@]11CC...,NaN,P06276,NaN,>lcl|BSEQ0016706|Cholinesterase\nMHSKVTIICIRFL...,Humans,1,True,True
24521,DB19353,Benzgalantamine,[H][C@]12C[C@@H](OC(=O)C3=CC=CC=C3)C=C[C@]11CC...,NaN,P36544,NaN,>lcl|BSEQ0004597|Neuronal acetylcholine recept...,Humans,1,True,True
24522,DB19353,Benzgalantamine,[H][C@]12C[C@@H](OC(=O)C3=CC=CC=C3)C=C[C@]11CC...,NaN,A9X444,NaN,>lcl|BSEQ0052493|Muscle nicotinic acetylcholin...,Humans,1,True,True
24523,DB19372,Zoxazolamine,NC1=NC2=CC(Cl)=CC=C2O1,C13841,Q9H2S1,NaN,>lcl|BSEQ0008406|Small conductance calcium-act...,Humans,1,True,True


In [ ]:
# Drop the helper columns as we dont need them anymore
df_drugbank = df_drugbank.drop(columns=["valid_smiles", "valid_protein"])


In [61]:
df_drugbank.shape

(24451, 9)

In [62]:
print("Unique drugs after cleaning:", df_drugbank["drugbank_id"].nunique())
print("Unique proteins after cleaning:", df_drugbank["protein_uniprot_id"].nunique())

#also count unique pairs of drug and protein
print("Unique drug-protein pairs after cleaning:", df_drugbank[["drugbank_id", "protein_uniprot_id"]].drop_duplicates().shape[0])   

Unique drugs after cleaning: 8407
Unique proteins after cleaning: 4822
Unique drug-protein pairs after cleaning: 23883


In [63]:
# we want to keep only the unique drug protein pairs
df_drugbank = df_drugbank.drop_duplicates(subset=['drugbank_id','protein_uniprot_id'])
df_drugbank.shape

(23883, 9)

In [65]:

print("DrugBank DTI dataset head:", df_drugbank.head())
print("Yamanishi merged DTI dataset head:", merged_yamanishi_df.head())


DrugBank DTI dataset head:   drugbank_id     drug_name  \
0     DB00006   Bivalirudin   
1     DB00014     Goserelin   
2     DB00014     Goserelin   
3     DB00014     Goserelin   
4     DB00027  Gramicidin D   

                                              smiles kegg_drug_id  \
0  CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...       D03136   
1  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...       D00573   
2  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...       D00573   
3  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...       D00573   
4  CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...       D04369   

  protein_uniprot_id  kegg_protein_id  \
0             P00734              NaN   
1             P22888              NaN   
2             P01148              NaN   
3             P30968              NaN   
4             P0AC13              NaN   

                                    protein_sequence  \
0  >lcl|BSEQ0016004|Prothrombin\nMAHVRGLQLPGCLALA...   
1  >lcl|BSEQ

In [71]:
#yamanishi columns are named wrongly, we will rename them


merged_yamanishi_df = merged_yamanishi_df.rename(columns={
    "drug_id": "kegg_protein_id",
    "target_id": "kegg_drug_id"
})


In [72]:
#make sure the KEGG columns are clean
df_drugbank["kegg_drug_id"] = df_drugbank["kegg_drug_id"].astype(str).str.strip()
df_drugbank["kegg_protein_id"] = df_drugbank["kegg_protein_id"].astype(str).str.strip()

merged_yamanishi_df["kegg_drug_id"] = merged_yamanishi_df["kegg_drug_id"].astype(str).str.strip()
merged_yamanishi_df["kegg_protein_id"] = merged_yamanishi_df["kegg_protein_id"].astype(str).str.strip()


/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_26683/1650716341.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drugbank["kegg_drug_id"] = df_drugbank["kegg_drug_id"].astype(str).str.strip()
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_26683/1650716341.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drugbank["kegg_protein_id"] = df_drugbank["kegg_protein_id"].astype(str).str.strip()


In [74]:
#check if df_drugbank["kegg_protein_id"] is all NaN
df_drugbank["kegg_protein_id"].unique() 
df_drugbank["kegg_drug_id"].unique() 

#check if there is any kegg drug id in drugbank that is nan
df_drugbank[df_drugbank["kegg_drug_id"].isna()]

,drugbank_id,drug_name,smiles,kegg_drug_id,protein_uniprot_id,kegg_protein_id,protein_sequence,organism,label


In [75]:
#since kegg_protein_id in drugbank is all NaN 
# we will transform kegg_protein_id in merged_yamanishi_df to uniprot ids
import pandas as pd
import requests
from tqdm import tqdm


def kegg_to_uniprot(kegg_ids, species="hsa"):
    """
    Maps KEGG gene/protein IDs to UniProt IDs using KEGG REST API.
    
    Example input: ["hsa:1956", "hsa:5290"]
    """
    base_url = f"https://rest.kegg.jp/conv/uniprot/{species}"
    
    response = requests.get(base_url)
    
    if response.status_code != 200:
        raise Exception("Failed to fetch KEGG-UniProt mapping table")

    mapping = {}
    
    for line in response.text.strip().split("\n"):
        kegg, uniprot = line.split("\t")
        kegg_id = kegg.replace("gene:", "")
        uniprot_id = uniprot.replace("uniprot:", "")
        mapping[kegg_id] = uniprot_id

    # Map only requested ids
    return {kid: mapping.get(kid, None) for kid in kegg_ids}


In [76]:
# Get unique KEGG protein IDs
kegg_protein_ids = merged_yamanishi_df["kegg_protein_id"].dropna().unique().tolist()

print("Total unique KEGG protein IDs:", len(kegg_protein_ids))


Total unique KEGG protein IDs: 989


In [77]:
kegg_uniprot_map = kegg_to_uniprot(kegg_protein_ids, species="hsa")

print("Example mappings:")
list(kegg_uniprot_map.items())[:10]


Example mappings:


[('hsa:190', 'up:F1D8P4'),
 ('hsa:2099', 'up:G4XH65'),
 ('hsa:2100', 'up:Q7LCB3'),
 ('hsa:2101', 'up:Q569H8'),
 ('hsa:2103', 'up:O95718'),
 ('hsa:2104', 'up:F1D8R5'),
 ('hsa:2908', 'up:F1D8N4'),
 ('hsa:3174', 'up:Q14541'),
 ('hsa:367', 'up:Q9NUA2'),
 ('hsa:4306', 'up:B0ZBF6')]

In [78]:
merged_yamanishi_df["uniprot_id"] = merged_yamanishi_df["kegg_protein_id"].map(kegg_uniprot_map)

# Check how many mapped successfully
print("Mapped UniProt IDs:", merged_yamanishi_df["uniprot_id"].notna().sum())
print("Unmapped IDs:", merged_yamanishi_df["uniprot_id"].isna().sum())


Mapped UniProt IDs: 5124
Unmapped IDs: 3


In [80]:
df_drugbank = df_drugbank.rename(columns={
    "protein_uniprot_id": "uniprot_id"
})


df_drugbank.columns, merged_yamanishi_df.columns

(Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id', 'uniprot_id',
        'kegg_protein_id', 'protein_sequence', 'organism', 'label'],
       dtype='object'),
 Index(['kegg_protein_id', 'kegg_drug_id', 'dataset', 'uniprot_id'], dtype='object'))

In [ ]:
#kegg_drug_id, uniprot_id -> drugbank drug-target pair
#kegg_drug_id, uniprot_id -> yamanishi drug-target pair

db = df_drugbank.copy()
yam = merged_yamanishi_df.copy()

#Build a set of Yamanishi pairs
#yamanishi_pairs = set(zip(yam["kegg_drug_id"], yam["uniprot_id"]))

#I need to first remove the up from yam["uniprot_id"] -> up:F1D8P4 - F1D8P4
yam["uniprot_id"] = yam["uniprot_id"].str.replace(r"^up:", "", regex=True)





In [94]:
yam['pair'] = list(zip(yam["kegg_drug_id"], yam["uniprot_id"]))


In [96]:
db.columns

Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id', 'uniprot_id',
       'kegg_protein_id', 'protein_sequence', 'organism', 'label'],
      dtype='object')

In [97]:
before = len(db)

# Create a boolean mask to identify DrugBank entries that are NOT in Yamanishi
# First, create a column with tuples for easy comparison
db['pair'] = list(zip(db["kegg_drug_id"], db["uniprot_id"]))

# Filter out entries that exist in Yamanishi pairs
db_filtered = db[~db['pair'].isin(yam['pair'])].copy()

# Drop the temporary pair column
db_filtered = db_filtered.drop('pair', axis=1)

print(f"DrugBank entries before filtering: {before}")
print(f"DrugBank entries after removing Yamanishi overlaps: {len(db_filtered)}")
print(f"Removed {before - len(db_filtered)} overlapping entries")


DrugBank entries before filtering: 23883
DrugBank entries after removing Yamanishi overlaps: 23736
Removed 147 overlapping entries


In [103]:
print("Yamanishi Shape:", yam.shape)
print("Filtered DrugBank Shape:", db_filtered.shape)

db_filtered.columns
yam.columns

Yamanishi Shape: (5127, 5)
Filtered DrugBank Shape: (23736, 9)


Index(['kegg_protein_id', 'kegg_drug_id', 'dataset', 'uniprot_id', 'pair'], dtype='object')

In [104]:
print("Unique drugs in filtered DrugBank:", db_filtered["drugbank_id"].nunique())
print("Unique proteins in filtered DrugBank:", db_filtered["uniprot_id"].nunique())

print("Unique drugs in filtered Yamanishi:", yam["kegg_drug_id"].nunique())
print("Unique proteins in filtered Yamanishi:", yam["uniprot_id"].nunique())


Unique drugs in filtered DrugBank: 8381
Unique proteins in filtered DrugBank: 4821
Unique drugs in filtered Yamanishi: 791
Unique proteins in filtered Yamanishi: 987


In [ ]:
yam.columns 
# we have to get smiles for yamanishi kegg drug ids to have a complete dataset
# uniprot_ids are fine

Index(['kegg_protein_id', 'kegg_drug_id', 'dataset', 'uniprot_id', 'pair'], dtype='object')

In [ ]:
db_filtered.columns # already have smiles and uniprot ids.

Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id', 'uniprot_id',
       'kegg_protein_id', 'protein_sequence', 'organism', 'label'],
      dtype='object')

In [107]:
# now we just have to save them as csvs
yam.to_csv(DATA_DIR_YAMANISHI / "yamanishi_final.csv", index=False)
db_filtered.to_csv(DATA_DIR_DRUGBANK / "drugbank_filtered_final.csv", index=False)

In [ ]:
#================================SKIP FROM THIS PART===============================

yamanishi_clean = merged_yamanishi_df.rename(columns={
    "drug_id": "kegg_drug_id"
})

yamanishi_clean = yamanishi_clean[[
    "kegg_drug_id",
    "uniprot_id"
]]

# Yamanishi is assumed positive interactions
yamanishi_clean["label"] = 1



In [ ]:

yamanishi_clean = yamanishi_clean.dropna(subset=["kegg_drug_id", "uniprot_id"])
df_drugbank = df_drugbank.dropna(subset=["kegg_drug_id", "uniprot_id"])
yamanishi_clean.shape, df_drugbank.shape

((5124, 3), (23883, 3))

In [46]:
union = pd.concat([
    df_drugbank,
    yamanishi_clean
], ignore_index=True)

# Deduplicate based on standardized IDs
union = union.drop_duplicates(subset=["kegg_drug_id", "uniprot_id"])

print("Final Union Dataset Shape:", union.shape)


Final Union Dataset Shape: (21053, 3)


In [47]:
# now we check the shapes

print("DrugBank DTI dataset shape:", df_drugbank.shape)
print("Yamanishi merged DTI dataset shape:", merged_yamanishi_df.shape)
print("Merged DrugBank and Yamanishi DTI dataset shape:", union.shape)

union.head()


DrugBank DTI dataset shape: (23883, 3)
Yamanishi merged DTI dataset shape: (5127, 4)
Merged DrugBank and Yamanishi DTI dataset shape: (21053, 3)


,kegg_drug_id,uniprot_id,label
0,D03136,P00734,1
1,D00573,P22888,1
2,D00573,P01148,1
3,D00573,P30968,1
4,D04369,P0AC13,1


In [48]:
print("Unique drugs:", union["kegg_drug_id"].nunique())
print("Unique proteins:", union["uniprot_id"].nunique())
print("Positive interactions:", union["label"].sum())


Unique drugs: 3239
Unique proteins: 5809
Positive interactions: 21053


In [49]:
#Find out the unique drugs and proteins in the each of the datasets
unique_drugs_drugbank = df_drugbank["kegg_drug_id"].nunique()
unique_proteins_drugbank = df_drugbank["uniprot_id"].nunique()
unique_drugs_yamanishi = yamanishi_clean["kegg_drug_id"].nunique()
unique_proteins_yamanishi = yamanishi_clean["uniprot_id"].nunique()
print("DrugBank - Unique Drugs:", unique_drugs_drugbank, "Unique Proteins:", unique_proteins_drugbank)
print("Yamanishi - Unique Drugs:", unique_drugs_yamanishi, "Unique Proteins:", unique_proteins_yamanishi)

DrugBank - Unique Drugs: 2579 Unique Proteins: 4822
Yamanishi - Unique Drugs: 791 Unique Proteins: 987


In [50]:
# count the duplicated rows for the union dataset
duplicated_rows = union.duplicated(subset=["kegg_drug_id", "uniprot_id"]).sum()
print("Number of duplicated drug-protein pairs in the union dataset:", duplicated_rows)

Number of duplicated drug-protein pairs in the union dataset: 0


**From the paper:**

The number of unique drugs in our **positive datasets** is **2,118**,comprising 
**1,328 from DrugBank and 790 from Yamanishi’s data**. In the same data, the
number of unique **drug-targets is 2,077 (706 from DrugBank and 1,371 from Yamanishi ).**
Finally, the number of known DTIs between the drugs and targets in the positive data is
**10,736 (3,530 from DrugBank and 7,206 from Yamanishi.)**

In [52]:
#convert the union into csv
DATA_DIR_POSITIVE_DTI = Path("../data/positive dti datasets")
union_csv_path = DATA_DIR_POSITIVE_DTI / "drugbank_yamanishi_merged.csv"
union.to_csv(union_csv_path, index=False)
